# GameTheory-04h — Jeux de routage / congestion & Price of Anarchy

**Navigation** : [<< 4g-Supermodularite](GameTheory-04g-Supermodularite-Python.ipynb) | [Index](README.md)

Ce carnet ouvre une branche manquante de la série : les **jeux de routage / congestion à flux fractionnaire** et le **price of anarchy (PoA)** — l'écart entre l'équilibre où chaque usager minimise son propre coût (équilibre de Wardrop, qui est un équilibre de Nash du jeu de routage) et l'**optimum social** qui minimise le coût total.

C'est un approfondissement du palier 04 (Équilibre de Nash) vu du côté de la **nature** de l'équilibre : au lieu de demander *si* un équilibre existe, on mesure *à quel point* il est efficace. Un petit réseau concret, des coûts de lien fonctions de la charge, un équilibre d'utilisateur calculé, un optimum social calculé, et le **PoA mesuré** sur l'instance. On s'appuie sur nos fondations **forme normale** (GameTheory-02 / -04) : l'équilibre de Wardrop est un équilibre de Nash.

**Références** : Roughgarden, *The Price of Anarchy of Network Games* (2005) · Roughgarden & Tardos, *How to Use Selfish Players to Design Efficient Algorithms* (2002 — la borne 4/3 pour les congestions linéaires/affines).

In [1]:
def cout_congestionne(x, alpha=1.0):
    """Cout par usager d'un lien congestionne : c(x) = alpha * x."""
    return alpha * x

def cout_libre(x):
    """Cout par usager d'un lien 'libre' (cout constant, sans congestion)."""
    return 1.0

print('Briques du modele chargees :')
print('  lien congestionne : c(x) = alpha * x')
print('  lien libre        : c(x) = 1')

Briques du modele chargees :
  lien congestionne : c(x) = alpha * x
  lien libre        : c(x) = 1


## 1. L'instance canonique : l'exemple de Pigou

Deux chemins en parallèle relient la source `s` à la destination `t`, avec une demande totale `D` d'usagers :

- **Autoroute** (lien congestionné) : le coût par usager est proportionnel au nombre d'usagers dessus, `c_A(x_A) = alpha * x_A`.
- **Route locale** (lien libre) : coût constant `c_B = 1`, sans congestion.

Chaque usager choisit un chemin. C'est l'exemple fondateur qui donne son nom au phénomène : la route locale gratuite conduit les usagers égoïstes à se jeter tous sur l'autoroute, alors que l'optimum social les répartirait.

In [2]:
alpha = 1.0   # pente de congestion de l'autoroute
D = 1.0       # demande totale (normalisee)

# OPTIMUM SOCIAL : min  alpha*x_A^2 + x_B   sous  x_A + x_B = D
#   (cout total = flux * cout par usager sur chaque lien)
#   condition du premier ordre : 2*alpha*x_A - 1 = 0  ->  x_A* = 1/(2*alpha)
xA_opt = 1.0 / (2.0 * alpha)
xB_opt = D - xA_opt
cout_opt = alpha * xA_opt**2 + xB_opt * 1.0

print(f"Optimum social : x_A = {xA_opt:.3f},  x_B = {xB_opt:.3f}")
print(f"Cout total a l'optimum  = {cout_opt:.4f}")

Optimum social : x_A = 0.500,  x_B = 0.500
Cout total a l'optimum  = 0.7500


## 2. L'équilibre de Wardrop (équilibre d'utilisateur / Nash)

À l'équilibre de Wardrop, **aucun usager ne peut réduire son coût personnel en changeant unilatéralement de chemin**. Tant que le coût de l'autoroute `c_A(x_A) = alpha * x_A` est inférieur au coût de la route locale (`1`), les usagers préfèrent l'autoroute : ils s'y massent jusqu'à ce que `alpha * x_A` atteigne `1`. Au-delà, plus personne n'y gagne à s'y ajouter.

Cet équilibre est un **équilibre de Nash** du jeu de routage : la stratégie de chaque joueur (son chemin) est une meilleure réponse à celle des autres.

In [3]:
# EQUILIBRE DE WARDROP : tous sur l'autoroute tant que c_A(D) = alpha*D <= 1
# (sinon l'autoroute coute plus que la route libre -> repartition jusqu'a c_A = 1)
if alpha * D <= 1.0:
    xA_ue, xB_ue = D, 0.0          # l'autoroute absorbe toute la demande
else:
    xA_ue = 1.0 / alpha            # repartition : c_A = c_libre = 1
    xB_ue = D - xA_ue

cout_ue = alpha * xA_ue**2 + xB_ue * 1.0

print(f"Equilibre de Wardrop : x_A = {xA_ue:.3f},  x_B = {xB_ue:.3f}")
print(f"Cout total a l'equilibre = {cout_ue:.4f}")

Equilibre de Wardrop : x_A = 1.000,  x_B = 0.000
Cout total a l'equilibre = 1.0000


## 3. Le Price of Anarchy — mesuré

Le **price of anarchy** rapporte le coût à l'équilibre égoïste au coût à l'optimum social :

$$\text{PoA} = \frac{\text{cout total à l'équilibre de Wardrop}}{\text{cout total à l'optimum social}}$$

C'est le chiffre central du carnet : il **quantifie** le prix de l'intérêt individuel décentralisé. PoA = 1 signifie que l'équilibre est déjà optimal ; plus il est grand, plus l'égoïsme collective est coûteux.

In [4]:
poa = cout_ue / cout_opt

print(f"Price of Anarchy = {cout_ue:.4f} / {cout_opt:.4f} = {poa:.4f}")
print(f"  -> l'egarement interesse coute {100*(poa - 1):.1f} % de cout supplementaire")
print(f"  -> l'exemple de Pigou donne PoA = 4/3 = {4/3:.4f}  (coherence : {abs(poa - 4/3) < 1e-9})")

Price of Anarchy = 1.0000 / 0.7500 = 1.3333
  -> l'egarement interesse coute 33.3 % de cout supplementaire
  -> l'exemple de Pigou donne PoA = 4/3 = 1.3333  (coherence : True)


## 4. La borne 4/3 (Roughgarden–Tardos, 2002)

**Théorème.** Pour un réseau à **une seule commodity** dont les coûts de lien sont **linéaires/affines** `c_e(x) = a_e x_e + b_e` (modèle à flux fractionnaire), le price of anarchy est **borné par 4/3** :

$$\text{PoA} \le 4/3$$

Cette borne est **serrée** : l'instance de Pigou mesurée ci-dessus la **atteint exactement** en `α = 1` (PoA = 4/3) — elle en est le **cas d'égalité**. C'est le chiffre central de la branche : l'écart maximal entre l'équilibre égoïste et l'optimum social, pour toute congestion linéaire, est de **33 %**. (Le « 5/2 » souvent cité relève des jeux de congestion **atomiques** — joueurs indivisibles — de Christodoulou–Koutsoupias, un modèle distinct de celui de ce carnet.)

> **Formalisation Lean de la borne 4/3** : prévue en suivi dédié (le carnet Python mesure ; la preuve formelle de la borne est portée par un carnet `*-Lean` compagnon). Elle n'est pas dans le périmètre de ce grain.

## 5. Sensibilité : le PoA en fonction de la pente de congestion

Faisons varier la pente `alpha` de l'autoroute (à demande `D = 1` fixe) et **mesurons** le PoA. On attend : plus la congestion est raide, plus l'écart se referme — le PoA **décroît** vers 1. Le maximum tombe en `alpha = 1`, où Pigou réalise l'égalité de la borne ; sur tout ce domaine le PoA mesuré doit rester **≤ 4/3**.

In [5]:
def poa_pigou(alpha, D=1.0):
    """PoA mesure de l'instance de Pigou (autoroute alpha*x, route libre 1, flux fractionnaire)."""
    # optimum social : x_A = 1/(2 alpha), cout total = D - 1/(4 alpha)
    xA_o = 1.0 / (2.0 * alpha)
    xB_o = D - xA_o
    cout_o = alpha * xA_o**2 + xB_o
    # equilibre de Wardrop : tous sur l'autoroute si alpha*D <= 1, sinon repartition
    if alpha * D <= 1.0:
        xA_u, xB_u = D, 0.0
    else:
        xA_u = 1.0 / alpha
        xB_u = D - 1.0 / alpha
    cout_u = alpha * xA_u**2 + xB_u
    return cout_u / cout_o

print(f"{'alpha':>6} | {'PoA mesure':>12} | {'<= 4/3 ?':>16}")
print("-" * 34)
for a in [1.0, 1.2, 1.4, 1.5, 1.8, 2.0]:
    p = poa_pigou(a)
    statut = "oui" if p <= 4.0/3.0 + 1e-9 else "non"
    print(f"{a:>6.1f} | {p:>12.4f} | {statut:>16}")

 alpha |   PoA mesure |         <= 4/3 ?
----------------------------------
   1.0 |       1.3333 |              oui
   1.2 |       1.2632 |              oui
   1.4 |       1.2174 |              oui
   1.5 |       1.2000 |              oui
   1.8 |       1.1613 |              oui
   2.0 |       1.1429 |              oui


La mesure confirme le profil théorique : le PoA est **maximal en `α = 1`** — valeur 4/3, le cas d'égalité de la borne — et **décroît** vers 1 quand la pente de congestion croît. La raison est directe : à `D = 1`, dès que `α > 1` l'équilibre se répartit et le coût d'équilibre est **figé à 1**, tandis que l'optimum social (`1 − 1/(4α)`) s'en approche. Tout le domaine reste donc **≤ 4/3**, jamais au-dessus : c'est exactement la borne de Roughgarden–Tardos, et le 4/3 n'est pas un artefact du point de bord — c'est la borne elle-même, atteinte en `α = 1`. Le PoA n'est pas un mot d'esprit : c'est une **quantité mesurable** qui structure la conception des réseaux.

## 6. Exercices

### Exercice 1 — Équilibre de Wardrop général

Pour une instance à **2 liens en parallèle** (autoroute `c(x) = alpha*x`, route libre `c = 1`) et une demande `D`, calcule l'équilibre de Wardrop, c'est-à-dire la répartition `(x_A, x_B)` des usagers à l'équilibre.

**Indice** : à l'équilibre, un usager ne gagne rien à changer de lien — compare le coût `c_A(x_A)` de l'autoroute au coût `1` de la route locale.

In [6]:
def wardrop_2liens(alpha, D):
    """Renvoie (x_A, x_B) a l'equilibre de Wardrop de l'instance a 2 liens.

    Autoroute : c(x) = alpha*x.  Route libre : c = 1.  Demande totale : D.
    """
    # TODO etudiant
    return None  # TODO: (x_A, x_B)

### Exercice 2 — La fonction PoA

Écris une fonction `poa(alpha, D)` qui renvoie le **price of anarchy mesuré** de l'instance de Pigou généralisée (autoroute `alpha*x`, route libre `1`, demande `D`). Elle doit reproduire la valeur `4/3` pour `alpha = 1, D = 1`.

In [7]:
def poa(alpha, D):
    """Renvoie le price of anarchy (cout Wardrop / cout optimum social).

    Instance : autoroute c(x) = alpha*x, route libre c = 1, demande D.
    Indice : recompose l'optimum social (FOC) et l'equilibre de Wardrop.
    """
    # TODO etudiant
    return None  # TODO: float

### Exercice 3 — Le PoA en fonction de la demande

Pour l'instance canonique (`alpha = 1`), calcule le PoA pour une **série de valeurs de la demande** `D` sur `[1, 4]` et décris la tendance. Que devient le PoA quand la demande croît ? (Rappel : à l'équilibre, tous sont sur l'autoroute tant que `alpha * D <= 1` ; au-delà, le flux se répartit.)

In [8]:
def poa_en_fonction_de_D(D, alpha=1.0):
    """Renvoie le PoA de l'instance canonique (alpha=1) pour une demande D."""
    # TODO etudiant
    return None  # TODO: float

# (a completer) evaluer sur D = [1.0, 2.0, 3.0, 4.0] et afficher la tendance

## Conclusion

Le **price of anarchy** transforme une intuition (« les usagers égoïstes gâchent le réseau ») en **quantité mesurable** : sur l'exemple de Pigou, l'intérêt individuel décentralisé coûte **33 %** de supplément (PoA = 4/3) par rapport à l'optimum social, et la borne 4/3 de Roughgarden–Tardos encadre ce prix pour toute la classe des congestions linéaires — Pigou en est le cas d'égalité. C'est le pont entre la **théorie des jeux** (l'équilibre de Nash / Wardrop) et la **conception d'algorithmes et de réseaux** efficaces — la branche appliquée que ce carnet ajoute à la série.